In [ ]:
import getpass
import sys

_USER = getpass.getuser()
sys.path.insert(0, f"/home/{_USER}/app/apps/apps/generator")

import tsgm  # noqa: E402, F401  -- must be imported before keras
import keras  # noqa: E402
import joblib  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402
import random

from genpm.modelling.core.artifacts import load_trained_model  # noqa: E402
from genpm.modelling.core.generation import generate_windows  # noqa: E402
from genpm.modelling.validation import (  # noqa: E402
    wide_timespan_to_long,
    plot_timeseries_overlay,
    plot_kde,
    plot_autocorr,
    plot_kpi_correlation,
)
from genpm.utils.consts import SHARED_DIR_PATH

from genpm.utils.spark_session import SparkDataManager  # noqa: E402
from genpm.preprocessing.logic.scaling import SimpleMinMaxScaler, YeoJohnsonScaler  # noqa: E402

In [ ]:
# ── Point this at whichever run you want to inspect ──
RUN_ID = "diffusion_run_5_yj"
WEIGHTS_FILE = "ddpm_5_yj.weights.h5"  # or the "*_last.weights.h5" fallback

RUN_DIR = SHARED_DIR_PATH / "model_runs" / RUN_ID
WEIGHTS_PATH = RUN_DIR / "models_weights_debug" / WEIGHTS_FILE

PREPROC_DIR = SHARED_DIR_PATH / "preprocessed_dataset" / "final_pmcm_yj"

model, config_encoder, cell_config_map = load_trained_model(
    run_id_path=RUN_DIR,
    weights_path=WEIGHTS_PATH,
)

X_scaled = np.load(RUN_DIR / "X_scaled.npy")
y_all = np.load(RUN_DIR / "y.npy")
cell_ids = np.load(RUN_DIR / "cell_ids.npy", allow_pickle=True)
window_anchors = np.load(RUN_DIR / "window_anchors.npy", allow_pickle=True)
kpi_columns = np.load(RUN_DIR / "kpi_columns.npy", allow_pickle=True).tolist()

SEQ_LEN, FEAT_DIM = X_scaled.shape[1], X_scaled.shape[2]
print(f"Loaded {RUN_ID}: {len(X_scaled):,} windows, seq_len={SEQ_LEN}, feat_dim={FEAT_DIM}")

## Pick a couple of configs to compare

One example real `distname` per *distinct* cell-config combination actually present in training. `generate_windows(cell_id=...)` looks the config up automatically from `cell_config_map`, and labels its output with a `cell_id` column — which `wide_timespan_to_long` renames to `distname`, keeping real and synthetic directly comparable.

In [ ]:
from collections import defaultdict

_by_combo = defaultdict(list)
for _cid, _cfg in cell_config_map["map"].items():
    _by_combo[tuple(_cfg)].append(_cid)

# label -> example real cell_id for that config
CONFIGS = {f"config_{i+1} ({combo[1]}/{combo[2]}/{combo[3]})": cids[0] for i, (combo, cids) in enumerate(_by_combo.items())}
print(f"{len(CONFIGS)} distinct configs found:")
for label, cid in CONFIGS.items():
    print(f"  {label}  ->  cell_id={cid}")

In [ ]:
# before we run it is advised to search for possible kpis to generate
# pd.set_option('display.max_rows', 500)
# defs = pd.read_parquet(PREPROC_DIR / "HELPER_kpi_definitions")
# defs[defs["kpi_id"].isin(kpi_columns)]

In [ ]:
# EXAMPLE_KPIS = ["NR_125", "NR_1479", "NR_135", "NR_1266", "NR_1225", "NR_5069", "NR_5110"]
EXAMPLE_KPIS = random.sample(kpi_columns, 30)
EXAMPLE_KPIS = [k for k in EXAMPLE_KPIS if k in kpi_columns]
kpi_idx = [kpi_columns.index(k) for k in EXAMPLE_KPIS]

## Helper functions

In [ ]:
def _to_numpy(t):
    try:
        return np.asarray(t)
    except TypeError:
        return t.detach().cpu().numpy()


def select_real_windows(cell_id: str, max_windows: int = 8, seed: int = 0):
    """Real (X, y, window_anchor) rows for one cell, capped to max_windows."""
    idx = np.where(cell_ids == cell_id)[0]
    if len(idx) == 0:
        raise ValueError(f"No windows found for cell_id={cell_id!r}")
    if len(idx) > max_windows:
        idx = np.random.default_rng(seed).choice(idx, size=max_windows, replace=False)
    return X_scaled[idx], y_all[idx], pd.to_datetime(window_anchors[idx])


def wide_from_array(arr: np.ndarray, anchors: pd.DatetimeIndex, cell_id: str) -> pd.DataFrame:
    """(n, seq_len, n_kpis) array -> wide df with cell_id/window_anchor/timestamp + KPI cols,
    same shape generate_windows() produces, so wide_timespan_to_long handles both identically."""
    n, seq_len, n_kpis = arr.shape
    flat = arr.reshape(n * seq_len, n_kpis)
    df = pd.DataFrame(flat, columns=kpi_columns[:n_kpis])
    df.insert(
        0,
        "timestamp",
        pd.to_datetime(np.repeat(anchors.values, seq_len))
        + pd.to_timedelta(np.tile(np.arange(seq_len), n), unit="h"),
    )
    df.insert(0, "window_anchor", np.repeat(anchors.values, seq_len))
    df.insert(0, "cell_id", cell_id)
    return df


def reconstruct(X_batch: np.ndarray, y_batch: np.ndarray) -> np.ndarray:
    """Encode real windows -> sample z from the posterior -> decode. True autoencoder
    reconstruction (distinct from generate(), which samples z from the prior N(0,1))."""
    x_hat = model([X_batch, y_batch], training=False)
    return _to_numpy(x_hat)

## Inverse transform back to physical KPI units

Load the saved `SimpleMinMaxScaler` / `YeoJohnsonScaler` params and wrap the exact inverse pipeline from `inverse_transform_testing.ipynb` (MinMax inverse → Yeo–Johnson inverse) in `inverse_to_physical(...)`. Both the real windows and the synthetic output come out of the model in `[0,1]` scaled space; this maps them back to original KPI units before the section-2 comparisons. The scaler params join on `kpi_id` + the five `[CELL] …` config columns, which we attach per cell from `cell_config_map`.

In [ ]:
# ── Inverse-transform setup (mirrors inverse_transform_testing.ipynb) ──
# Scaler params saved at preprocessing time, keyed by kpi_id + the five [CELL] config
# columns. Repoint PREPROC_DIR if the run was preprocessed into a different dataset.


sdm = SparkDataManager()
mm_params = sdm.read_parquet(PREPROC_DIR / "scaling_params_df")
yj_params = sdm.read_parquet(PREPROC_DIR / "yj_scaling_params_df")

_MM_GROUP_COLS = [c for c in mm_params.columns if c not in ("mm_min", "mm_max")]
_YJ_GROUP_COLS = [c for c in yj_params.columns if c != "yj_lambda"]

_mm_scaler = SimpleMinMaxScaler.load_params(
    mm_params, value_col="kpi_value", group_cols=_MM_GROUP_COLS
)
_yj_scaler = YeoJohnsonScaler.load_params(
    yj_params, value_col="kpi_value", group_cols=_YJ_GROUP_COLS
)
print(f"Inverse-transform params loaded — group cols: {_MM_GROUP_COLS}")


def _attach_config_cols(long_df: pd.DataFrame, cell_id: str) -> pd.DataFrame:
    """Add the five [CELL] config columns the scaler params join on, for this cell."""
    cfg_cols = cell_config_map["config_cols"]
    cfg_vals = cell_config_map["map"][str(cell_id)]
    out = long_df[["distname", "timestamp", "window_anchor", "kpi_id", "kpi_value"]].copy()
    for col, val in zip(cfg_cols, cfg_vals, strict=True):
        out[col] = val
    return out


def inverse_to_physical(long_real: pd.DataFrame, long_syn: pd.DataFrame, cell_id: str):
    """Scaled-space long frames -> original physical KPI units.

    Runs the same MinMax-inverse -> Yeo-Johnson-inverse pipeline as
    inverse_transform_testing.ipynb. Real and synthetic share this cell's config, so
    they are inverted together in one Spark job (tagged with __src__) and split back.
    """
    real = _attach_config_cols(long_real, cell_id).assign(__src__="real")
    syn = _attach_config_cols(long_syn, cell_id).assign(__src__="syn")
    combined = pd.concat([real, syn], ignore_index=True)

    sdf = sdm.spark.createDataFrame(combined)
    sdf = _mm_scaler.inverse_transform(sdf)  # undo MinMax [0,1] scaling
    sdf = _yj_scaler.inverse_transform(sdf)  # undo Yeo-Johnson
    inv = sdf.toPandas()

    keep = ["distname", "timestamp", "window_anchor", "kpi_id", "kpi_value"]
    real_phys = inv.loc[inv["__src__"] == "real", keep].reset_index(drop=True)
    syn_phys = inv.loc[inv["__src__"] == "syn", keep].reset_index(drop=True)
    return real_phys, syn_phys

## Generation vs real (physical units) — KDE / autocorrelation / cross-KPI / mean±1σ

`generate_windows` samples and decodes conditioned on `y` — the actual synthetic-data-generation path. Both the decoded synthetic windows and the real windows are then mapped back to original KPI units with `inverse_to_physical(...)`, and compared distributionally (not paired) against real data for the same cell. All plots below are in physical KPI units.

IMPORTANT NOTE Generation fo constant and changing regime KPIs is not realiable need more robust filtering before modelling

In [ ]:
N_WINDOWS = 8      # real windows / synthetic weeks sampled per config
ANCHOR_DATE = "2024-01-15"
# CORR_KPIS = kpi_columns  # full set for the cross-KPI heatmap (labels auto-hidden if >30)
CORR_KPIS = EXAMPLE_KPIS

for label, cid in CONFIGS.items():
    print(f"\n========== {label}  (cell_id={cid}) ==========")

    X_real, _, anchors_real = select_real_windows(cid, max_windows=N_WINDOWS)
    long_real_scaled = wide_timespan_to_long(wide_from_array(X_real, anchors_real, cid))

    syn_wide = generate_windows(
        model=model,
        config_encoder=config_encoder,
        cell_config_map=cell_config_map,
        cell_id=cid,
        anchor_date=ANCHOR_DATE,
        n_weeks=N_WINDOWS,
        holiday=0,
        batch_size=64,
        seed=0,
        kpi_list=kpi_columns,
    )
    long_syn_scaled = wide_timespan_to_long(syn_wide)

    # Map both real and synthetic back to original, physical KPI units before comparing.
    long_real, long_syn = inverse_to_physical(long_real_scaled, long_syn_scaled, cid)

    plot_timeseries_overlay(long_real, long_syn, EXAMPLE_KPIS, cid, ANCHOR_DATE,
                             str(pd.Timestamp(ANCHOR_DATE) + pd.Timedelta(weeks=N_WINDOWS)))
    plot_kde(long_real, long_syn, EXAMPLE_KPIS)
    plot_autocorr(long_real, long_syn, EXAMPLE_KPIS)
    plot_kpi_correlation(long_real, long_syn, CORR_KPIS)